# 01. LangChain tool call


> 랭체인에서도 tool 사용 가능

In [1]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

llm = ChatOpenAI(model="gpt-4o-mini")

llm.invoke([HumanMessage("하이~")])

AIMessage(content='안녕하세요! 어떻게 도와드릴까요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 10, 'total_tokens': 20, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_520fb14a9c', 'id': 'chatcmpl-DuuNekFkHYpJ6A9TzeH6Hlnqvz7ZW', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--019f029f-f863-7ce0-b593-cae42aad54e3-0', usage_metadata={'input_tokens': 10, 'output_tokens': 10, 'total_tokens': 20, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [2]:
from langchain_core.tools import tool

In [4]:
from datetime import datetime
import pytz

@tool # @Lang Chian에 tool을 등록하는 과정 tool 데코레이터를 사용하여 함수를 도구로 등록
def get_current_time(timezone: str, location: str) -> str:
    ### 단순한 주석이 아니라 랭체인에 이 함수의 기능과 입력값, 사용 방법을 알려주는 문서화 문자열
    """ 현재 시각을 반환하는 함수 

    Args:
        timezone (str): 타임존 (예: 'Asia/Seoul') 실제 존재하는 타임존이어야 함
        location (str): 지역명. 타임존이 모든 지명에 대응되지 않기 때문에 이후 llm 답변 생성에 사용됨
    """
    tz = pytz.timezone(timezone)
    now = datetime.now(tz).strftime("%Y-%m-%d %H:%M:%S")
    location_and_local_time = f'{timezone} ({location}) 현재시각 {now} ' # 타임존, 지역명, 현재시각을 문자열로 반환
    print(location_and_local_time)
    return location_and_local_time


In [5]:
# 도구를 tools 리스트에 추가하고, tool_dict에도 추가
tools = [get_current_time,]
tool_dict = {"get_current_time": get_current_time,}

In [7]:
# 도구를 모델에 바인딩: 모델에 도구를 바인딩하면, 도구를 사용하여 llm 답변을 생성할 수 있음
llm_with_tools = llm.bind_tools(tools)

In [23]:
llm_with_tools

RunnableBinding(bound=ChatOpenAI(client=<openai.resources.chat.completions.completions.Completions object at 0x00000241D62EC250>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x00000241D6358610>, root_client=<openai.OpenAI object at 0x00000241D62EC520>, root_async_client=<openai.AsyncOpenAI object at 0x00000241D6358670>, model_name='gpt-4o-mini', model_kwargs={}, openai_api_key=SecretStr('**********'), openai_proxy=None, stream_usage=True), kwargs={'tools': [{'type': 'function', 'function': {'name': 'get_current_time', 'description': "현재 시각을 반환하는 함수 \n\n   Args:\n       timezone (str): 타임존 (예: 'Asia/Seoul') 실제 존재하는 타임존이어야 함\n       location (str): 지역명. 타임존이 모든 지명에 대응되지 않기 때문에 이후 llm 답변 생성에 사용됨", 'parameters': {'properties': {'timezone': {'type': 'string'}, 'location': {'type': 'string'}}, 'required': ['timezone', 'location'], 'type': 'object'}}}]}, config={}, config_factories=[])

In [8]:
from langchain_core.messages import SystemMessage

# (4) 사용자의 질문과 tools 사용하여 llm 답변 생성
messages = [
    SystemMessage("너는 사용자의 질문에 답변을 하기 위해 tools를 사용할 수 있다."),
    HumanMessage("서울은 지금 몇시야?"),
]

# (5) llm_with_tools를 사용하여 사용자의 질문에 대한 llm 답변 생성
response = llm_with_tools.invoke(messages)
print(response)
messages.append(response)

content='' additional_kwargs={'tool_calls': [{'id': 'call_1QWmXT9Lj2t2w07VDhF4h7V1', 'function': {'arguments': '{"timezone":"Asia/Seoul","location":"서울"}', 'name': 'get_current_time'}, 'type': 'function'}], 'refusal': None} response_metadata={'token_usage': {'completion_tokens': 22, 'prompt_tokens': 134, 'total_tokens': 156, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_b4b9deb02a', 'id': 'chatcmpl-DuuSKGU84HByFWZSK5CUSpcBfT0QY', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None} id='run--019f02a4-6428-7752-b27e-a8a1dab2d96f-0' tool_calls=[{'name': 'get_current_time', 'args': {'timezone': 'Asia/Seoul', 'location': '서울'}, 'id': 'call_1QWmXT9Lj2t2w07VDhF4h7V1', 'type': 'tool_call'}] usage_metadata={'input_tokens': 134, 'output_tokens': 22, '

In [9]:
messages

[SystemMessage(content='너는 사용자의 질문에 답변을 하기 위해 tools를 사용할 수 있다.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='서울은 지금 몇시야?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_1QWmXT9Lj2t2w07VDhF4h7V1', 'function': {'arguments': '{"timezone":"Asia/Seoul","location":"서울"}', 'name': 'get_current_time'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 22, 'prompt_tokens': 134, 'total_tokens': 156, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_b4b9deb02a', 'id': 'chatcmpl-DuuSKGU84HByFWZSK5CUSpcBfT0QY', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--019f02a4-6428-7752-b27e-a8a1dab2d96f-0', tool_cal

In [10]:
response.tool_calls

[{'name': 'get_current_time',
  'args': {'timezone': 'Asia/Seoul', 'location': '서울'},
  'id': 'call_1QWmXT9Lj2t2w07VDhF4h7V1',
  'type': 'tool_call'}]

In [14]:
tool_call = response.tool_calls[0]
tool_call

{'name': 'get_current_time',
 'args': {'timezone': 'Asia/Seoul', 'location': '서울'},
 'id': 'call_1QWmXT9Lj2t2w07VDhF4h7V1',
 'type': 'tool_call'}

In [12]:
tool_call["name"]

'get_current_time'

In [ ]:
tool_dict[tool_call["name"]] #StructuredTool : Langchain 객체 -> invoke 진행 필요

StructuredTool(name='get_current_time', description="현재 시각을 반환하는 함수 \n\n   Args:\n       timezone (str): 타임존 (예: 'Asia/Seoul') 실제 존재하는 타임존이어야 함\n       location (str): 지역명. 타임존이 모든 지명에 대응되지 않기 때문에 이후 llm 답변 생성에 사용됨", args_schema=<class 'langchain_core.utils.pydantic.get_current_time'>, func=<function get_current_time at 0x00000241D84B78B0>)

In [16]:
tool_dict

{'get_current_time': StructuredTool(name='get_current_time', description="현재 시각을 반환하는 함수 \n\n   Args:\n       timezone (str): 타임존 (예: 'Asia/Seoul') 실제 존재하는 타임존이어야 함\n       location (str): 지역명. 타임존이 모든 지명에 대응되지 않기 때문에 이후 llm 답변 생성에 사용됨", args_schema=<class 'langchain_core.utils.pydantic.get_current_time'>, func=<function get_current_time at 0x00000241D84B78B0>)}

In [ ]:
tool_dict[tool_call["name"]].invoke(tool_call)  #decorate가 없으면 일반함수라서 invoke가 안됨

Asia/Seoul (서울) 현재시각 2026-06-26 15:39:52 


ToolMessage(content='Asia/Seoul (서울) 현재시각 2026-06-26 15:39:52 ', name='get_current_time', tool_call_id='call_1QWmXT9Lj2t2w07VDhF4h7V1')

In [18]:
for tool_call in response.tool_calls:
    selected_tool = tool_dict[tool_call["name"]] # (7) tool_dict를 사용하여 도구 함수를 선택
    print(tool_call["args"]) # (8) 도구 호출 시 전달된 인자 출력
    tool_msg = selected_tool.invoke(tool_call) # (9) 도구 함수를 호출하여 결과를 반환
    messages.append(tool_msg)

messages

{'timezone': 'Asia/Seoul', 'location': '서울'}
Asia/Seoul (서울) 현재시각 2026-06-26 15:42:12 


[SystemMessage(content='너는 사용자의 질문에 답변을 하기 위해 tools를 사용할 수 있다.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='서울은 지금 몇시야?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_1QWmXT9Lj2t2w07VDhF4h7V1', 'function': {'arguments': '{"timezone":"Asia/Seoul","location":"서울"}', 'name': 'get_current_time'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 22, 'prompt_tokens': 134, 'total_tokens': 156, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_b4b9deb02a', 'id': 'chatcmpl-DuuSKGU84HByFWZSK5CUSpcBfT0QY', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--019f02a4-6428-7752-b27e-a8a1dab2d96f-0', tool_cal

In [20]:
llm_with_tools.invoke(messages)

AIMessage(content='서울은 지금 2026년 6월 26일 15시 42분 12초입니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 25, 'prompt_tokens': 189, 'total_tokens': 214, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_b4b9deb02a', 'id': 'chatcmpl-DuuaRKwNLjLYyhr0Bnb14ltF5wpX4', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--019f02ac-128d-70c3-95d6-215f1f455385-0', usage_metadata={'input_tokens': 189, 'output_tokens': 25, 'total_tokens': 214, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [21]:
messages.append(llm_with_tools.invoke(messages))

In [22]:
messages

[SystemMessage(content='너는 사용자의 질문에 답변을 하기 위해 tools를 사용할 수 있다.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='서울은 지금 몇시야?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_1QWmXT9Lj2t2w07VDhF4h7V1', 'function': {'arguments': '{"timezone":"Asia/Seoul","location":"서울"}', 'name': 'get_current_time'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 22, 'prompt_tokens': 134, 'total_tokens': 156, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_b4b9deb02a', 'id': 'chatcmpl-DuuSKGU84HByFWZSK5CUSpcBfT0QY', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--019f02a4-6428-7752-b27e-a8a1dab2d96f-0', tool_cal

### 주가 예제

In [24]:
import yfinance as yf

@tool
def get_yf_stock_history(ticker: str, period: str) -> str:
    """ 
    주식 종목의 가격 데이터를 조회하는 함수

    Args:
        ticker (str): 주식 종목 코드 (예: AAPL)
        period (str): 주식 데이터 조회 기간 (예: 1d, 1mo, 1y)
    
    """
    stock = yf.Ticker(ticker)
    history = stock.history(period=period)
    history_md = history.to_markdown() 

    return history_md

tools = [get_current_time, get_yf_stock_history]
tool_dict = {"get_current_time": get_current_time, "get_yf_stock_history": get_yf_stock_history}

llm_with_tools = llm.bind_tools(tools)

In [25]:
messages

[SystemMessage(content='너는 사용자의 질문에 답변을 하기 위해 tools를 사용할 수 있다.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='서울은 지금 몇시야?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_1QWmXT9Lj2t2w07VDhF4h7V1', 'function': {'arguments': '{"timezone":"Asia/Seoul","location":"서울"}', 'name': 'get_current_time'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 22, 'prompt_tokens': 134, 'total_tokens': 156, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_b4b9deb02a', 'id': 'chatcmpl-DuuSKGU84HByFWZSK5CUSpcBfT0QY', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--019f02a4-6428-7752-b27e-a8a1dab2d96f-0', tool_cal

In [26]:
messages.append(HumanMessage("테슬라는 한달 전에 비해 주가가 올랐나 내렸나?"))

response = llm_with_tools.invoke(messages)
messages.append(response)

In [27]:
response.tool_calls

[{'name': 'get_yf_stock_history',
  'args': {'ticker': 'TSLA', 'period': '1mo'},
  'id': 'call_xWhoQ9n8BoYlY7Jus2djIrYV',
  'type': 'tool_call'}]

In [30]:
!pip install tabulate

In [32]:
#### 이어서 자연어로 모델이 답변하는 것 추가 실습
tool_call = response.tool_calls[0] #dict 형태로 변경
selected_tool = tool_dict[tool_call["name"]] #StructuredTool : Langchain 객체 -> invoke 진행 필요

for tool_call in response.tool_calls:
    selected_tool = tool_dict[tool_call["name"]] # (7) tool_dict를 사용하여 도구 함수를 선택
selected_tool

StructuredTool(name='get_yf_stock_history', description='주식 종목의 가격 데이터를 조회하는 함수\n\nArgs:\n    ticker (str): 주식 종목 코드 (예: AAPL)\n    period (str): 주식 데이터 조회 기간 (예: 1d, 1mo, 1y)', args_schema=<class 'langchain_core.utils.pydantic.get_yf_stock_history'>, func=<function get_yf_stock_history at 0x00000241D84D10D0>)

In [31]:
#### 이어서 자연어로 모델이 답변하는 것 추가 실습
tool_call = response.tool_calls[0] #dict 형태로 변경
selected_tool = tool_dict[tool_call["name"]] #StructuredTool : Langchain 객체 -> invoke 진행 필요

for tool_call in response.tool_calls:
    selected_tool = tool_dict[tool_call["name"]] # (7) tool_dict를 사용하여 도구 함수를 선택
    print(tool_call["args"]) # (8) 도구 호출 시 전달된 인자 출력
    tool_msg = selected_tool.invoke(tool_call) # (9) 도구 함수를 호출하여 결과를 반환
    messages.append(tool_msg)

llm_with_tools.invoke(messages)

{'ticker': 'TSLA', 'period': '1mo'}


AIMessage(content='한 달 전, 2026년 5월 26일 테슬라(TSLA)의 종가는 433.59 달러였고, 현재(2026년 6월 25일) 종가는 375.12 달러입니다. \n\n따라서, 테슬라의 주가는 한 달 전에 비해 내렸습니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 75, 'prompt_tokens': 1668, 'total_tokens': 1743, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_fff1d3b4b6', 'id': 'chatcmpl-Duuo3emhg9S6Xxj6Rtuu7ofO68Ho7', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--019f02b8-f43c-7b32-8556-2749f84d4f92-0', usage_metadata={'input_tokens': 1668, 'output_tokens': 75, 'total_tokens': 1743, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

## 실습 1. pdf_summary.py

pdf_summary 하는 함수 tool로 등록하여 사용해보기

+ summarize_text 도 langchain 사용하는 것으로 변경

In [33]:
from langchain_core.messages import HumanMessage
import os 

pdf_path = os.path.join(os.getcwd(),"samples/videoanomaly.pdf")
messages = [
    HumanMessage(f"이 {pdf_path} 문서의 저자는 누구야?"),
    ]



In [ ]:
from summary_answer import summarize_pdf 


In [39]:
tool_list = [summarize_pdf]
tool_dict = {'summarize_pdf':summarize_pdf}


In [40]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model = 'gpt-4o-mini')

In [41]:
llm_with_tools = llm.bind_tools(tool_list)

In [42]:
response = llm_with_tools.invoke(messages)

In [43]:
messages.append(response)

In [44]:
for tool_call in response.tool_calls:
    out= tool_dict[tool_call['name']].invoke(tool_call)

In [45]:
messages.append(out)

In [46]:
llm_with_tools.invoke(messages)

AIMessage(content='문서의 저자는 Qianzi Yu, Kai Zhu, Yang Cao, Yu Kang입니다. 이들은 중국의 과학기술대학교 및 헌페이 종합 국가 과학 센터의 연구원들로, 인공지능 및 비디오 이상 탐지 분야에서 연구하고 있습니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 63, 'prompt_tokens': 557, 'total_tokens': 620, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_bbc23ca3d7', 'id': 'chatcmpl-DuvNdazW5iSJicLbhcFRQj5a0bpAY', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--019f02da-9b9c-7950-8d77-cfcc1bfa287c-0', usage_metadata={'input_tokens': 557, 'output_tokens': 63, 'total_tokens': 620, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})